In [ ]:
%pip install transformers datasets torch --quiet

In [ ]:
import pandas as pd
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import matplotlib.pyplot as plt

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset, ClassLabel

In [ ]:
train = pd.read_csv("../dataimport/train.csv")
valid = pd.read_csv("../dataimport/valid.csv")
test = pd.read_csv("../dataimport/test.csv")

print("Train :", train.shape)
print("Valid :", valid.shape)
print("Test  :", test.shape)

Train : (10240, 14)
Valid : (1284, 14)
Test  : (1267, 14)


statement_len_words : Nombre de mots dans le statement
statement_len_chars : Nombre de caractères dans le statement

In [ ]:
train["statement_len_words"] = train["statement"].astype(str).apply(lambda x: len(x.split()))
valid["statement_len_words"] = valid["statement"].astype(str).apply(lambda x: len(x.split()))
test["statement_len_words"] = test["statement"].astype(str).apply(lambda x: len(x.split()))

train["statement_len_chars"] = train["statement"].astype(str).apply(len)
valid["statement_len_chars"] = valid["statement"].astype(str).apply(len)
test["statement_len_chars"] = test["statement"].astype(str).apply(len)

nb_exclamation : Nombre de points d'exclamation !
nb_question : Nombre de points d'interrogation ?

In [ ]:
train["nb_exclamation"] = train["statement"].astype(str).apply(lambda x: x.count("!"))
valid["nb_exclamation"] = valid["statement"].astype(str).apply(lambda x: x.count("!"))
test["nb_exclamation"] = test["statement"].astype(str).apply(lambda x: x.count("!"))

train["nb_question"] = train["statement"].astype(str).apply(lambda x: x.count("?"))
valid["nb_question"] = valid["statement"].astype(str).apply(lambda x: x.count("?"))
test["nb_question"] = test["statement"].astype(str).apply(lambda x: x.count("?"))

Creation len word / nb_exclamation	/ nb_question

In [ ]:
train.head()

,id,label,statement,subject,speaker,speaker_job_title,state_info,party_affiliation,barely_true_counts,false_counts,half_true_counts,mostly_true_counts,pants_on_fire_counts,context,statement_len_words,statement_len_chars,nb_exclamation,nb_question
0,2635.json,false,Says the Annies List political group supports ...,abortion,dwayne-bohac,State representative,Texas,republican,0.0,1.0,0.0,0.0,0.0,a mailer,11,82,0,0
1,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0.0,0.0,1.0,1.0,0.0,a floor speech.,24,141,0,1
2,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70.0,71.0,160.0,163.0,9.0,Denver,19,105,0,0
3,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting,NaN,NaN,none,7.0,19.0,3.0,5.0,44.0,a news release,12,78,0,0
4,9028.json,half-true,The economic turnaround started at the end of ...,"economy,jobs",charlie-crist,NaN,Florida,democrat,15.0,9.0,20.0,19.0,2.0,an interview on CNN,10,54,0,0


In [ ]:
count_cols = [
    "barely_true_counts",
    "false_counts",
    "half_true_counts",
    "mostly_true_counts",
    "pants_on_fire_counts"
]

for df in [train, valid, test]:
    for col in count_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    df["history_total"] = df[count_cols].sum(axis=1)
    df["history_false_ratio"] = (df["false_counts"] + df["pants_on_fire_counts"]) / (df["history_total"] + 1)

^ passif du locuteur history_total	

In [ ]:
df.head() 

,id,label,statement,subject,speaker,speaker_job_title,state_info,party_affiliation,barely_true_counts,false_counts,half_true_counts,mostly_true_counts,pants_on_fire_counts,context,statement_len_words,statement_len_chars,nb_exclamation,nb_question,history_total,history_false_ratio
0,11972.json,true,Building a wall on the U.S.-Mexico border will...,immigration,rick-perry,Governor,Texas,republican,30,30,42,23,18,Radio interview,11,68,0,0,143,0.333333
1,11685.json,false,Wisconsin is on pace to double the number of l...,jobs,katrina-shankland,State representative,Wisconsin,democrat,2,1,0,0,0,a news conference,12,63,0,0,3,0.250000
2,11096.json,false,Says John McCain has done nothing to help the ...,"military,veterans,voting-record",donald-trump,President-Elect,New York,republican,63,114,51,37,61,comments on ABC's This Week.,10,51,0,0,326,0.535168
3,5209.json,half-true,Suzanne Bonamici supports a plan that will cut...,"medicare,message-machine-2012,campaign-adverti...",rob-cornilles,consultant,Oregon,republican,1,1,3,1,1,a radio show,13,85,0,0,7,0.250000
4,9524.json,pants-fire,When asked by a reporter whether hes at the ce...,"campaign-finance,legal-issues,campaign-adverti...",state-democratic-party-wisconsin,NaN,Wisconsin,democrat,5,7,2,2,7,a web video,23,127,0,0,23,0.583333


In [ ]:
def normalize_text(t):
    return str(t).strip().lower()  # customiser
for df in [train, valid, test]:
    df["statement_clean"] = df["statement"].astype(str).apply(normalize_text)